In [1]:
from braket.program_sets import ProgramSet
from mpqp import QCircuit
from mpqp.core.languages import Language
from mpqp.gates import *
from mpqp.tools.circuit import random_circuit

cs = []

for _ in range(5):
    cs.append(random_circuit().to_other_language(Language.BRAKET))

s = ProgramSet(cs)

In [2]:
from braket.devices import LocalSimulator

device = LocalSimulator()
result_1 = device.run(s, shots=300).result()

In [3]:
for r in result_1:
    for ex in r:
        print(len(r))
        print(ex.counts)

1
Counter({'01000': 60})
1
Counter({'00000': 60})
1
Counter({'00000': 19, '00011': 19, '01011': 11, '01000': 11})
1
Counter({'10010': 55, '00010': 5})
1
Counter({'00101': 58, '00100': 2})


In [4]:
from sympy import Symbol
from braket.program_sets import CircuitBinding
from braket.devices import LocalSimulator

t = Symbol("t")
r = Symbol("r")

c1 = QCircuit([Rz(t, 0), Rx(r, 0), Rz(t, 0)]).to_other_language(Language.BRAKET)

cb = CircuitBinding(c1, input_sets={"t": (0, 1), "r": (1, 0.01)})
s1 = ProgramSet(cb, 100)
device = LocalSimulator()
result_2 = device.run(s1).result()
for r in result_2:
    for rr in r:
        print(rr.counts)

Counter({'0': 76, '1': 24})
Counter({'0': 100})


In [ ]:
from mpqp.core.circuit import BindingMode, CircuitBinding as MPQPBinding
from mpqp import QCircuit, Language
from mpqp.core.instruction.measurement.expectation_value import ExpectationMeasure
from mpqp.execution.job import JobType
from braket.program_sets import ProgramSet


def test(bind: MPQPBinding, programSet: bool = True) -> ProgramSet | MPQPBinding:
    from braket.program_sets import ProgramSet
    from braket.circuits import Circuit

    # translate inner circuits to braket and CB's elements to Braket
    translated: list[MPQPBinding | Circuit] = []
    for c in bind.circuits:
        if isinstance(c, QCircuit):
            translated.append(c.to_other_language(Language.BRAKET))
        else:
            translation = test(c, False)

            if isinstance(translation, list):
                translated.extend(translation)
            else:
                assert isinstance(translation, MPQPBinding)
                translated.append(translation)

    # translate var
    var = {}
    if bind.value:
        var = bind.value
        converted = {}
        for key in var.keys():
            val = var[key]
            if any([not isinstance(v, float | int) for v in val]):
                raise ValueError(f"Cannot use complex parameters in Braket. Got: {val}")
            converted.update({str(key): tuple(val)})
        var = converted

    from braket.program_sets import CircuitBinding

    obs = []
    if bind.measurements:
        for m in bind.measurements:
            assert isinstance(m, ExpectationMeasure)
            obs.extend([o.to_other_language(Language.BRAKET) for o in m.observables])

    if not programSet:  # If the circuitBinding is embedded store the translated data.
        bind._translated_circuits = translated
        bind._translated_observables = obs
        bind._translated_variables = var
        return bind

    result = []
    if bind.mode == BindingMode.ZIP:
        if len(translated) == 1:
            circuit = translated[0]
            if isinstance(circuit, MPQPBinding):
                if circuit.mode == BindingMode.ZIP:
                    assert circuit._translated_circuits and circuit._translated_observables and circuit._translated_variables
                    observables = obs+list(circuit._translated_observables)
                    variables: dict[str, float] = var | dict(circuit._translated_variables)    
                    if not (len(observables) == 0 or len(variables) == 0): # Full Zip1-Zip1 and Zip1-Zip2 
                        if len(circuit.circuits) == 1:
                            return ProgramSet.zip(CircuitBinding(circuit._translated_circuits[0], input_sets=variables), observables=observables, shots_per_executable=bind.shots)
                        else:
                            return ProgramSet.zip(circuit._translated_observables, input_sets=variables, observables=observables, shots_per_executable=bind.shots) 
                    else: # Multiple circuit within a single ZIP bind (Partial Zip1-Zip1 or Zip1-Zip2)
                        if len(observables) == 0:
                            if len(circuit._translated_circuits) == 1:
                                return ProgramSet(CircuitBinding(circuit._translated_circuits[0], input_sets=[variables]), shots_per_executable=bind.shots)
                            else:
                                return ProgramSet.zip(circuit._translated_circuits, input_sets=variables, shots_per_executable=bind.shots)
                        elif len(variables) == 0:
                            if len(circuit._translated_circuits) == 1:
                                result.append(CircuitBinding(circuit._translated_circuits[0], input_sets=[variables]))
                            else:
                                result.extend([CircuitBinding(c, observables=[observables[i]]) for i,c in enumerate(circuit._translated_circuits)])
                        


                            

                        
                else: # ZIP1 - ZIP2 CB(CB([c,c1], [p1,p2], ZIP), [o1,o2], ZIP) ==> {c,p1,o1} + {c1,p2,o2}
                    assert circuit._translated_circuits and circuit._translated_observables and circuit._translated_variables
                    observables: list[Circuit] = obs+circuit._translated_observables
                    variables: dict[str, float] = var | circuit._translated_variables
                    if len(observables) == 0: # CB([c,CB([c1,c2], [p,p2])]) ==> {c} + {c1, p} + {c2, p2}

                    elif len(variables) == 0: # CB([c,CB(c2,[o1,o2])]) ==> {c} + {c2, o1} + {c2, o2}

                    else:
                        if len(circuit.circuits) != len(observables) or len(circuit.circuits) != len(variables):
                            raise ValueError("Number of circuits must be exactly 1 or the same as the number of observables and variables.")
                        return ProgramSet.zip(circuits=circuit._translated_circuits, input_sets=variables, observables=observables, shots_per_executable=bind.shots)

            else:
                return ProgramSet.zip(CircuitBinding(circuit, input_sets=var), observables=obs, shots_per_executable=bind.shots)
        elif all([isinstance(c,Circuit) for c in translated]):
            # need to convert dict of list to list of dicts (ask braket)
            v = [{}]
            for key in var.keys():
                for i,el in enumerate(var[key]):
                    if i == len(v):
                        v.append({})
                    params = v[i]
                    params.update({key: el})

            return ProgramSet.zip(translated, input_sets=v, observables=obs, shots_per_executable=bind.shots) # type: ignore
        else:
            print("caca")
    if programSet:
        from braket.program_sets import ProgramSet

        return ProgramSet(result, bind.shots)
    else:
        return result

In [ ]:
from sympy import Symbol
from mpqp.core.circuit import CircuitBinding as MPQPBinding
from braket.devices import LocalSimulator
from mpqp.core.instruction.measurement.expectation_value import Observable
from mpqp.core.instruction.measurement.pauli_string import pI, pX, pZ
from mpqp.gates import *
from braket.program_sets import ProgramSet, CircuitBinding

t = Symbol("t")
r = Symbol("r")

nb_param = 2
nb_circuits = 2
nb_obs = 3

m = ExpectationMeasure([Observable(pZ), Observable(pX), Observable(pI)], [0])
c1 = QCircuit([Ry(t, 0), Rx(r, 0)])  # .to_other_language(Language.BRAKET)
c = QCircuit([Ry(1, 0), Rx(0.4, 0)])  # .to_other_language(Language.BRAKET)
p = {t: [0, 1], r: [1, 0]}
test = MPQPBinding([MPQPBinding(c1, p)], measurements=[m])


m = [
    Observable(pZ).to_other_language(Language.BRAKET),
    Observable(pI).to_other_language(Language.BRAKET),
]
"""
mb = ProgramSet([c, CircuitBinding(c1, input_sets=p, observables=m)], 100)
mm = ProgramSet.product([c, CircuitBinding(c1, observables=[m[0]])], observables=[m[1]])

device = LocalSimulator()
result_2 = device.run(mb).result()
for r in result_2:  # type: ignore
    print(len(r))
    """

ValueError: your circuit already contains measurements, you cannot have multiple measurements

c : 1

c1 : ([0,1] Z, [0,1] X, [0,1] I : [1,0.01] Z, [1,0.01] X, [1,0.01] I ) = 6
7 total runs.

In [51]:
p = [{"t": 0, "r": 1}, {"t": 1, "r": 0}]
m = [
    Observable(pZ).to_other_language(Language.BRAKET),
    # Observable(pI).to_other_language(Language.BRAKET),
]
mb = ProgramSet.zip(
    circuits=[c1.to_other_language(Language.BRAKET)],
    input_sets=p,
    observables=m,
    shots_per_executable=100,
)

device = LocalSimulator()
result_2 = device.run(mb).result()
for r in result_2:  # type: ignore
    for rr in r:
        print(rr.inputs)
        print(rr.observable)
        print(rr.counts)

ValueError: Number of circuits must match number of input sets

In [49]:
mb = ProgramSet(
    [
        CircuitBinding(
            c1.to_other_language(Language.BRAKET), input_sets=[p], observables=m
        ),
        CircuitBinding(
            c.to_other_language(Language.BRAKET),
            input_sets=[p],
            observables=m,
        ),
    ],
    shots_per_executable=100,
)
device = LocalSimulator()
result_2 = device.run(mb).result()
for r in result_2:  # type: ignore
    for rr in r:
        print(rr.inputs)
        print(rr.observable)
        print(rr.counts)

{'t': 0, 'r': 1, '_OBSERVABLE_THETA_0': 0, '_OBSERVABLE_PHI_0': 0, '_OBSERVABLE_OMEGA_0': 0}
Z('qubit_count': 1, 'target': QubitSet([Qubit(0)]))
Counter({'0': 79, '1': 21})
{'t': 0, 'r': 1, '_OBSERVABLE_THETA_0': 0, '_OBSERVABLE_PHI_0': 0, '_OBSERVABLE_OMEGA_0': 0}
I('qubit_count': 1, 'target': QubitSet([Qubit(0)]))
Counter({'0': 78, '1': 22})
{'t': 0, 'r': 1, '_OBSERVABLE_THETA_0': 0, '_OBSERVABLE_PHI_0': 0, '_OBSERVABLE_OMEGA_0': 0}
Z('qubit_count': 1, 'target': QubitSet([Qubit(0)]))
Counter({'0': 74, '1': 26})
{'t': 0, 'r': 1, '_OBSERVABLE_THETA_0': 0, '_OBSERVABLE_PHI_0': 0, '_OBSERVABLE_OMEGA_0': 0}
I('qubit_count': 1, 'target': QubitSet([Qubit(0)]))
Counter({'0': 65, '1': 35})


In [11]:
from mpqp.execution.devices import AWSDevice
from mpqp.execution.runner import run
from mpqp.core.instruction.measurement.expectation_value import Observable
from mpqp.core.instruction.measurement.pauli_string import pI, pX, pZ
from mpqp.gates import *
from mpqp import QCircuit, ExpectationMeasure

x = 0
y = 1
c = QCircuit(
    [Rz(x, 0), Rx(y, 0), Rz(x, 0), ExpectationMeasure(Observable(pX), shots=100)]
)

run(c, AWSDevice.BRAKET_LOCAL_SIMULATOR)

hfzehijfe
[0.46, 0.54]


Result(Job(JobType.OBSERVABLE, QCircuit([Rz(0, 0), Rx(1, 0), Rz(0, 0), ExpectationMeasure(Observable(pX, 'observable_0'), shots=100)]), AWSDevice.BRAKET_LOCAL_SIMULATOR), np.float64(-0.08000000000000002), None, 100)

In [31]:
a = {"a": 2, "b": 14}
b = {"b": 3}
#
print(a | b)

{'a': 2, 'b': 3}
{'a': 2, 'b': 14}
